**Import libraries**

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
import holidays

pd.set_option("display.max_columns", None)

**Config: where fitted objects and feature tables get saved**

In [ ]:
artifacts_dir = Path("feature_engineering_artifacts")
artifacts_dir.mkdir(exist_ok=True)

**Read artifacts: train, validation, and test**


In [ ]:
train_df = pd.read_parquet("train.parquet")
val_df = pd.read_parquet("val.parquet")
test_df = pd.read_parquet("test.parquet")

print("train:", train_df.shape, " val:", val_df.shape, " test:", test_df.shape)

**Create features**

In [ ]:
def add_derived_features(df):
    feat = df.copy()
    feat["order_purchase_timestamp"] = pd.to_datetime(feat["order_purchase_timestamp"])

    # same_state: at least one seller in the same state as the customer
    def _same_state(row):
        if pd.isna(row["seller_states"]) or pd.isna(row["customer_state"]):
            return np.nan
        seller_state_list = [s.strip() for s in str(row["seller_states"]).split(",")]
        return row["customer_state"] in seller_state_list

    feat["same_state"] = feat.apply(_same_state, axis=1)

    # purchase_month / purchase_weekday
    feat["purchase_month"] = feat["order_purchase_timestamp"].dt.month
    feat["purchase_weekday"] = feat["order_purchase_timestamp"].dt.day_name()

    # is_holiday: purchase date is a Brazilian public holiday
    years = feat["order_purchase_timestamp"].dt.year.unique().tolist()
    br_holidays = holidays.Brazil(years=years)
    feat["is_holiday"] = feat["order_purchase_timestamp"].dt.normalize().isin(br_holidays)
    feat["is_holiday"] = feat["is_holiday"].astype(bool)

    return feat


train_feat = add_derived_features(train_df)
val_feat = add_derived_features(val_df)
test_feat = add_derived_features(test_df)

**Shipping-pressure features (trailing 5-day window)**

Two versions, per the requirement:
1. `seller_pressure_5d` — load at the **seller-state** level using `seller_states`,
   because the feature table does not contain individual seller IDs. An order with
   several seller states gets the **mean** pressure across its states.
2. `customer_state_pressure_5d` — load at the **customer_state** level.

Both are **causal**: for order at time *t*, we count only orders placed in
`[t - 5 days, t)` — i.e. strictly *before* this order, never after. This avoids
look-ahead leakage.

Since `train_df` / `val_df` / `test_df` are a **random** split (not time-based),
the trailing count is computed over the **full combined order log**
(train+val+test) sorted by real purchase time, then mapped back to each split.
Only `order_purchase_timestamp`, `customer_state`, and `seller_states` are used;
none of these depend on the delivery outcome, so this does not leak the label.

In [ ]:
WINDOW_DAYS = 5


def _trailing_causal_count(times: np.ndarray, window_days: int) -> np.ndarray:
    """Count earlier timestamps in the trailing time window."""
    order = np.argsort(times)
    sorted_times = times[order]
    window = np.timedelta64(window_days, "D")

    upper = np.searchsorted(sorted_times, sorted_times, side="left")
    lower = np.searchsorted(sorted_times, sorted_times - window, side="left")
    sorted_counts = upper - lower

    counts = np.empty_like(sorted_counts)
    counts[order] = sorted_counts
    return counts


def add_shipping_pressure_features(train_df, val_df, test_df, window_days=WINDOW_DAYS):
    def prep(df, split_name):
        tmp = df[["order_purchase_timestamp", "customer_state", "seller_states"]].copy()
        tmp["order_purchase_timestamp"] = pd.to_datetime(tmp["order_purchase_timestamp"])
        tmp["_split"] = split_name
        tmp["_orig_index"] = df.index.values
        return tmp

    combined = pd.concat(
        [prep(train_df, "train"), prep(val_df, "val"), prep(test_df, "test")],
        ignore_index=True,
    )
    combined["_row_id"] = combined.index

    # Customer pressure grouped by customer state.
    combined["customer_state_pressure_5d"] = np.nan
    for state, group in combined.dropna(subset=["customer_state"]).groupby("customer_state"):
        counts = _trailing_causal_count(group["order_purchase_timestamp"].values, window_days)
        combined.loc[group.index, "customer_state_pressure_5d"] = counts

    # Seller pressure grouped by seller state.
    exploded = combined[["_row_id", "order_purchase_timestamp", "seller_states"]].copy()
    exploded["seller_states"] = exploded["seller_states"].fillna("")
    exploded = exploded.assign(
        seller_state=exploded["seller_states"].astype(str).str.split(",")
    ).explode("seller_state")
    exploded["seller_state"] = exploded["seller_state"].str.strip()
    exploded = exploded[exploded["seller_state"] != ""].reset_index(drop=True)

    exploded["seller_pressure_5d"] = np.nan
    for seller_state, group in exploded.groupby("seller_state"):
        counts = _trailing_causal_count(group["order_purchase_timestamp"].values, window_days)
        exploded.loc[group.index, "seller_pressure_5d"] = counts

    # Orders with multiple seller states get their mean pressure.
    seller_pressure_per_order = exploded.groupby("_row_id")["seller_pressure_5d"].mean()
    combined["seller_pressure_5d"] = combined["_row_id"].map(seller_pressure_per_order)

    def extract(split_name, orig_df):
        sub = combined[combined["_split"] == split_name].set_index("_orig_index")
        out = orig_df.copy()
        out["customer_state_pressure_5d"] = sub["customer_state_pressure_5d"]
        out["seller_pressure_5d"] = sub["seller_pressure_5d"]
        return out

    return extract("train", train_df), extract("val", val_df), extract("test", test_df)


train_feat_p, val_feat_p, test_feat_p = add_shipping_pressure_features(
    train_feat, val_feat, test_feat, window_days=WINDOW_DAYS
 )
train_feat["customer_state_pressure_5d"] = train_feat_p["customer_state_pressure_5d"]
train_feat["seller_pressure_5d"] = train_feat_p["seller_pressure_5d"]
val_feat["customer_state_pressure_5d"] = val_feat_p["customer_state_pressure_5d"]
val_feat["seller_pressure_5d"] = val_feat_p["seller_pressure_5d"]
test_feat["customer_state_pressure_5d"] = test_feat_p["customer_state_pressure_5d"]
test_feat["seller_pressure_5d"] = test_feat_p["seller_pressure_5d"]

print(train_feat[["customer_state_pressure_5d", "seller_pressure_5d"]].describe())

**Select features**

In [ ]:
## Final feature selection
numeric_features = [
    "total_price",
    "total_freight",
    "num_items",
    "num_sellers",
    "num_products",
    "total_weight",
    "num_payment_sequential",
    "avg_distance_km",
    "max_distance_km",
    "purchase_month",
    "customer_state_pressure_5d",
    "seller_pressure_5d",
]
categorical_single_features = [
    "customer_state",
    "purchase_weekday",
]
multi_value_feature = "payment_types"
boolean_features = ["same_state", "is_holiday"]

selected_features = (
    numeric_features + categorical_single_features + [multi_value_feature] + boolean_features
)
target = "is_late"

with open(artifacts_dir / "feature_list_raw.json", "w") as f:
    json.dump(selected_features, f, indent=2)

**Build the raw feature table for each split**

In [ ]:
train_table = train_feat[selected_features + [target]].copy()
val_table = val_feat[selected_features + [target]].copy()
test_table = test_feat[selected_features + [target]].copy()

print("Selected feature table shapes -- train:", train_table.shape, " val:", val_table.shape, " test:", test_table.shape)
print("\nFeatures ({}):".format(len(selected_features)), selected_features)
print("\nMissing values in train:")
print(train_table.isna().sum()[train_table.isna().sum() > 0])

train_table.head(20)

**Handle missing values, encode categories**

In [ ]:
# ---- Numeric columns: fit the imputer on train only ----
numeric_imputer = SimpleImputer(strategy="median")
train_table[numeric_features] = numeric_imputer.fit_transform(train_table[numeric_features])
val_table[numeric_features] = numeric_imputer.transform(val_table[numeric_features])
test_table[numeric_features] = numeric_imputer.transform(test_table[numeric_features])

# ---- same_state: fill value learned from train only ----
for table in (train_table, val_table, test_table):
    table["same_state_missing"] = table["same_state"].isna().astype(int)

same_state_fill_value = train_table["same_state"].mode(dropna=True)[0]
train_table["same_state"] = train_table["same_state"].fillna(same_state_fill_value)
val_table["same_state"] = val_table["same_state"].fillna(same_state_fill_value)
test_table["same_state"] = test_table["same_state"].fillna(same_state_fill_value)

# ---- categorical / multi-value columns: constant fill, no fitting needed ----
for table in (train_table, val_table, test_table):
    for col in categorical_single_features:
        table[col] = table[col].fillna("missing")
    table[multi_value_feature] = table[multi_value_feature].fillna("missing")

In [ ]:
# ---- One-hot encoding: fit on train only ----
onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
onehot_encoder.fit(train_table[categorical_single_features])

def onehot_transform(table):
    array = onehot_encoder.transform(table[categorical_single_features])
    cols = onehot_encoder.get_feature_names_out(categorical_single_features)
    return pd.DataFrame(array, columns=cols, index=table.index)

train_onehot_df = onehot_transform(train_table)
val_onehot_df = onehot_transform(val_table)
test_onehot_df = onehot_transform(test_table)

# ---- Multi-hot encoding for payment_types: fit on train only ----
def to_payment_lists(table):
    return table[multi_value_feature].apply(lambda x: [p.strip() for p in str(x).split(",")])

payment_encoder = MultiLabelBinarizer()
payment_encoder.fit(to_payment_lists(train_table))

def payment_transform(table):
    # unseen payment types in val/test are safely ignored by the already-fitted encoder
    array = payment_encoder.transform(to_payment_lists(table))
    cols = [f"payment_type_{c}" for c in payment_encoder.classes_]
    return pd.DataFrame(array, columns=cols, index=table.index)

train_payment_df = payment_transform(train_table)
val_payment_df = payment_transform(val_table)
test_payment_df = payment_transform(test_table)

# ---- booleans ----
bool_features_all = boolean_features + ["same_state_missing"]
train_bool_df = train_table[bool_features_all].astype(int)
val_bool_df = val_table[bool_features_all].astype(int)
test_bool_df = test_table[bool_features_all].astype(int)

**Assemble the final feature tables**

In [ ]:
def assemble(numeric_table, onehot_df, payment_df, bool_df, target_table):
    return pd.concat(
        [
            numeric_table[numeric_features].reset_index(drop=True),
            onehot_df.reset_index(drop=True),
            payment_df.reset_index(drop=True),
            bool_df.reset_index(drop=True),
            target_table[[target]].reset_index(drop=True),
        ],
        axis=1,
    )

final_train = assemble(train_table, train_onehot_df, train_payment_df, train_bool_df, train_table)
final_val = assemble(val_table, val_onehot_df, val_payment_df, val_bool_df, val_table)
final_test = assemble(test_table, test_onehot_df, test_payment_df, test_bool_df, test_table)

final_feature_list = [c for c in final_train.columns if c != target]

print("Final shapes -- train:", final_train.shape, " val:", final_val.shape, " test:", final_test.shape)
print("Final feature count:", len(final_feature_list))
print("Any NaNs left? train:", final_train.isna().sum().sum(),
      " val:", final_val.isna().sum().sum(),
      " test:", final_test.isna().sum().sum())

# val/test must have exactly the same columns as train, in the same order
assert list(final_val.columns) == list(final_train.columns), "val columns must match train exactly"
assert list(final_test.columns) == list(final_train.columns), "test columns must match train exactly"

final_train.head(20)

**Artifact: feature tables, fitted transformers, and feature list**

In [ ]:
# feature tables
final_train.to_parquet("feature_table_train.parquet", index=False)
final_val.to_parquet("feature_table_val.parquet", index=False)
final_test.to_parquet("feature_table_test.parquet", index=False)

# fitted transformers
joblib.dump(numeric_imputer, artifacts_dir / "numeric_imputer.joblib")
joblib.dump(onehot_encoder, artifacts_dir / "onehot_encoder.joblib")
joblib.dump(payment_encoder, artifacts_dir / "payment_encoder.joblib")
joblib.dump(same_state_fill_value, artifacts_dir / "same_state_fill_value.joblib")

# feature list actually used by the model (post encoding)
with open(artifacts_dir / "final_feature_list.json", "w") as f:
    json.dump(final_feature_list, f, indent=2)

print(f"Saved feature tables and fitted objects to: {artifacts_dir.resolve()}")